# Lesson 5a — House price predictor

A bridge between `pragma_mini.py` (3 keys) and the streaming churn predictor (4 keys × 15 events).
**One record per "user"**, **10 attributes**, **5 price classes**.

We're going to apply the PRAGMA recipe step by step. **First we look at the data**, then build intuition for how the model sees it, then run two experiments and compare.

## 🧰 Lesson reference legend

- **L1** — the 5-line training loop
- **L1b** — architecture vs training
- **L1c** — gradient descent details
- **L2** — tokens & embeddings
- **L3** — attention
- **L4** — masked language modelling
- **L5** — `pragma_mini.py`


## 0 — Imports and seeds

In [ ]:
import random
import copy
import torch
import torch.nn as nn

torch.manual_seed(0)
random.seed(0)

## 1 — Build your eye: look at the data first

Before any model, let's look at a few houses. Each house has 10 attributes.

In [ ]:
KEYS = ["bedrooms", "bathrooms", "size", "age", "neighborhood",
        "garage", "pool", "garden", "schools", "condition"]

VALUE_BUCKETS = {
    "bedrooms":     ["1bed", "2bed", "3bed", "4bed", "5+bed"],
    "bathrooms":    ["1bath", "2bath", "3+bath"],
    "size":         ["small", "medium", "large", "huge"],
    "age":          ["new", "modern", "older", "vintage"],
    "neighborhood": ["downtown", "suburb", "rural", "beach"],
    "garage":       ["nogarage", "1car", "2car"],
    "pool":         ["nopool", "haspool"],
    "garden":       ["nogarden", "smallgarden", "largegarden"],
    "schools":      ["poorschool", "avgschool", "goodschool", "excschool"],
    "condition":    ["poorcond", "faircond", "goodcond", "exccond"],
}

PAD, MASK = "<pad>", "<mask>"
VALUES = [v for vs in VALUE_BUCKETS.values() for v in vs]
vocab  = [PAD, MASK] + KEYS + VALUES
tok2id = {t: i for i, t in enumerate(vocab)}
V      = len(vocab)
PRICE_CLASSES = ["bargain", "cheap", "average", "expensive", "luxury"]

### Define how houses are generated — with **correlations**

The trick: each house belongs to an *archetype* (rural cottage, luxury beach, etc.). The archetype is a weighted distribution over each attribute. Beach houses tend to have pools. Vintage houses tend to be smaller. This is what gives MLM something to learn.

In [ ]:
ARCHETYPES = {
    "rural_cottage": {
        "bedrooms":     {"1bed":2, "2bed":5, "3bed":3, "4bed":1, "5+bed":0},
        "bathrooms":    {"1bath":6, "2bath":3, "3+bath":1},
        "size":         {"small":5, "medium":4, "large":1, "huge":0},
        "age":          {"new":0, "modern":1, "older":4, "vintage":5},
        "neighborhood": {"downtown":0, "suburb":1, "rural":8, "beach":1},
        "garage":       {"nogarage":3, "1car":5, "2car":2},
        "pool":         {"nopool":9, "haspool":1},
        "garden":       {"nogarden":1, "smallgarden":3, "largegarden":6},
        "schools":      {"poorschool":4, "avgschool":4, "goodschool":2, "excschool":0},
        "condition":    {"poorcond":2, "faircond":4, "goodcond":3, "exccond":1},
    },
    "suburban_family": {
        "bedrooms":     {"1bed":0, "2bed":1, "3bed":5, "4bed":3, "5+bed":1},
        "bathrooms":    {"1bath":1, "2bath":6, "3+bath":3},
        "size":         {"small":1, "medium":5, "large":3, "huge":1},
        "age":          {"new":2, "modern":5, "older":2, "vintage":1},
        "neighborhood": {"downtown":1, "suburb":7, "rural":1, "beach":1},
        "garage":       {"nogarage":1, "1car":3, "2car":6},
        "pool":         {"nopool":7, "haspool":3},
        "garden":       {"nogarden":1, "smallgarden":5, "largegarden":4},
        "schools":      {"poorschool":0, "avgschool":3, "goodschool":5, "excschool":2},
        "condition":    {"poorcond":0, "faircond":2, "goodcond":5, "exccond":3},
    },
    "urban_apartment": {
        "bedrooms":     {"1bed":4, "2bed":4, "3bed":2, "4bed":0, "5+bed":0},
        "bathrooms":    {"1bath":5, "2bath":4, "3+bath":1},
        "size":         {"small":7, "medium":2, "large":1, "huge":0},
        "age":          {"new":2, "modern":5, "older":2, "vintage":1},
        "neighborhood": {"downtown":9, "suburb":1, "rural":0, "beach":0},
        "garage":       {"nogarage":7, "1car":2, "2car":1},
        "pool":         {"nopool":9, "haspool":1},
        "garden":       {"nogarden":8, "smallgarden":2, "largegarden":0},
        "schools":      {"poorschool":1, "avgschool":3, "goodschool":4, "excschool":2},
        "condition":    {"poorcond":1, "faircond":3, "goodcond":4, "exccond":2},
    },
    "luxury_beach": {
        "bedrooms":     {"1bed":0, "2bed":1, "3bed":2, "4bed":4, "5+bed":3},
        "bathrooms":    {"1bath":0, "2bath":2, "3+bath":8},
        "size":         {"small":0, "medium":1, "large":4, "huge":5},
        "age":          {"new":5, "modern":4, "older":1, "vintage":0},
        "neighborhood": {"downtown":0, "suburb":0, "rural":0, "beach":10},
        "garage":       {"nogarage":0, "1car":1, "2car":9},
        "pool":         {"nopool":1, "haspool":9},
        "garden":       {"nogarden":0, "smallgarden":2, "largegarden":8},
        "schools":      {"poorschool":0, "avgschool":1, "goodschool":3, "excschool":6},
        "condition":    {"poorcond":0, "faircond":0, "goodcond":3, "exccond":7},
    },
    "old_townhouse": {
        "bedrooms":     {"1bed":1, "2bed":4, "3bed":4, "4bed":1, "5+bed":0},
        "bathrooms":    {"1bath":4, "2bath":5, "3+bath":1},
        "size":         {"small":2, "medium":5, "large":2, "huge":1},
        "age":          {"new":0, "modern":1, "older":4, "vintage":5},
        "neighborhood": {"downtown":5, "suburb":4, "rural":1, "beach":0},
        "garage":       {"nogarage":4, "1car":4, "2car":2},
        "pool":         {"nopool":9, "haspool":1},
        "garden":       {"nogarden":4, "smallgarden":5, "largegarden":1},
        "schools":      {"poorschool":2, "avgschool":4, "goodschool":3, "excschool":1},
        "condition":    {"poorcond":3, "faircond":4, "goodcond":2, "exccond":1},
    },
}

ARCHETYPE_WEIGHTS = {"rural_cottage":2, "suburban_family":4, "urban_apartment":2,
                    "luxury_beach":1, "old_townhouse":2}

PRICE_WEIGHTS = {
    "bedrooms":     {"1bed":0,"2bed":30,"3bed":60,"4bed":90,"5+bed":120},
    "bathrooms":    {"1bath":0,"2bath":25,"3+bath":55},
    "size":         {"small":0,"medium":80,"large":160,"huge":260},
    "age":          {"new":60,"modern":30,"older":0,"vintage":20},
    "neighborhood": {"downtown":150,"suburb":50,"rural":0,"beach":200},
    "garage":       {"nogarage":0,"1car":20,"2car":50},
    "pool":         {"nopool":0,"haspool":30},
    "garden":       {"nogarden":0,"smallgarden":15,"largegarden":40},
    "schools":      {"poorschool":0,"avgschool":40,"goodschool":90,"excschool":150},
    "condition":    {"poorcond":-50,"faircond":0,"goodcond":50,"exccond":110},
}
PRICE_BASE = 100

def weighted_choice(weights_dict):
    items, weights = zip(*weights_dict.items())
    return random.choices(items, weights=weights, k=1)[0]

def random_house():
    arch = weighted_choice(ARCHETYPE_WEIGHTS)
    return {k: weighted_choice(ARCHETYPES[arch][k]) for k in KEYS}

def house_price(h):
    p = PRICE_BASE
    for k, v in h.items():
        p += PRICE_WEIGHTS[k][v]
    if h["neighborhood"] == "beach" and h["pool"] == "haspool": p += 80
    if h["age"] == "vintage" and h["condition"] == "poorcond":  p -= 60
    if h["size"] == "huge" and h["bedrooms"] == "5+bed":        p += 60
    p += random.gauss(0, 30)
    return max(0, p)

def price_to_class(price):
    if price < 250: return 0
    if price < 450: return 1
    if price < 700: return 2
    if price < 1000: return 3
    return 4

### Look at three houses side by side

Notice how attributes within a house tend to **agree**. The luxury beach house has BIG values everywhere; the rural cottage has SMALL values everywhere. This is the correlation structure pre-training will learn.

In [ ]:
sample_arch = ["rural_cottage", "suburban_family", "luxury_beach"]
samples = []
for arch in sample_arch:
    dists = ARCHETYPES[arch]
    h = {k: weighted_choice(dists[k]) for k in KEYS}
    p = house_price(h)
    samples.append((arch, h, p, price_to_class(p)))

# Print side by side
header = f"  {'attribute':<14s}" + "".join(f"{a[0]:>16s}" for a in samples)
print(header)
print("  " + "-" * (14 + 16 * len(samples)))
for k in KEYS:
    print(f"  {k:<14s}" + "".join(f"{a[1][k]:>16s}" for a in samples))
print(f"  {'PRICE ($k)':<14s}" + "".join(f"{a[2]:>16.0f}" for a in samples))
print(f"  {'CLASS':<14s}" + "".join(f"{PRICE_CLASSES[a[3]]:>16s}" for a in samples))

## 2 — How the computer sees each house

The computer doesn't see a nice table. It sees a **flat list of token IDs**: 10 (key, value) pairs = 20 tokens per house.

In [ ]:
def encode_house(house):
    ids = []
    for k in KEYS:
        ids.append(tok2id[k])
        ids.append(tok2id[house[k]])
    return ids

first_arch, first_h, first_p, _ = samples[0]
print(f"House: {first_arch}")
print(f"\nDict form: {first_h}")
print(f"\nToken IDs: {encode_house(first_h)}")
print(f"\nTokens:    {[vocab[i] for i in encode_house(first_h)]}")
print(f"\nThe model never sees 'rural cottage' as a label. Just this 20-element list.")

## 3 — The TWO games the model plays

**This is the part that needs to be crystal clear.** Pre-training and the downstream task are *two separate training sessions* on the same data.

### Game 1: Pre-training (self-supervised)
- **Task**: fill-in-the-blank on house attributes (mask values, predict them)
- **Data**: ALL 8,000 houses
- **Labels needed?** No — the data labels itself (we just hide tokens we already have)
- **Goal**: teach the encoder the correlation structure

### Game 2: Downstream task (the thing we actually care about)
- **Task**: predict price class (bargain / cheap / avg / expensive / luxury)
- **Data**: a SMALL labelled set (50, 100, 500, or 4000 houses)
- **Labels needed?** Yes — we need the price for each training house
- **Goal**: classify houses by price

> The big PRAGMA bet: if Game 1 has taught the encoder something useful about house structure, Game 2 can be done with very few labels.

## 4 — Build the full dataset

Generate 8,000 houses for pre-training, with their price labels reserved for later.

In [ ]:
N_HOUSES = 8000

houses, classes = [], []
for _ in range(N_HOUSES):
    h = random_house()
    p = house_price(h)
    houses.append(h)
    classes.append(price_to_class(p))

X = torch.tensor([encode_house(h) for h in houses], dtype=torch.long)
y = torch.tensor(classes, dtype=torch.long)

print(f"shape X: {tuple(X.shape)}  ({X.size(1)} tokens per house)")
for c in range(5):
    print(f"  {PRICE_CLASSES[c]:>10s}: {int((y == c).sum())} houses")

## 5 — Architecture (L1b + L2 + L3)

Same backbone as `pragma_mini.py`. Embedding + position + 2 attention layers. The encoder is reusable across tasks — only the head changes.

In [ ]:
D_MODEL, N_HEADS, N_LAYERS = 32, 2, 2

class Encoder(nn.Module):
    def __init__(self, V, d=D_MODEL, heads=N_HEADS, layers=N_LAYERS, max_len=64):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.pos = nn.Embedding(max_len, d)
        layer    = nn.TransformerEncoderLayer(d, heads, d*2, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, layers)
    def forward(self, x):
        pos = torch.arange(x.size(1))
        return self.enc(self.emb(x) + self.pos(pos))

class MLMHead(nn.Module):
    def __init__(self, V, d=D_MODEL):
        super().__init__()
        self.proj = nn.Linear(d, V)
    def forward(self, h):
        return self.proj(h)

class PriceHead(nn.Module):
    def __init__(self, d=D_MODEL, n_classes=5):
        super().__init__()
        self.proj = nn.Linear(d, n_classes)
    def forward(self, h):
        return self.proj(h.mean(dim=1))   # average across positions, then project

print(f"Encoder knobs: {sum(p.numel() for p in Encoder(V).parameters()):,}")
print(f"MLMHead knobs: {sum(p.numel() for p in MLMHead(V).parameters()):,}")
print(f"PriceHead knobs: {sum(p.numel() for p in PriceHead().parameters())}")

## 6 — Pre-training (Game 1, L4)

For each batch:
1. Take a random batch of houses.
2. Mask ~30% of their VALUE tokens (never key tokens — those tell the model what KIND of thing to predict).
3. Ask the model to predict the masked values.
4. Compute loss, backprop, nudge knobs.
5. Repeat 3000 times.

The price labels are never used here.

In [ ]:
KEY_IDS = torch.tensor([tok2id[k] for k in KEYS])

def mlm_mask(X_batch, p=0.30):
    X = X_batch.clone()
    y = torch.full_like(X, -100)
    is_value = ~torch.isin(X, KEY_IDS)
    pick = (torch.rand_like(X, dtype=torch.float) < p) & is_value
    y[pick] = X[pick]
    X[pick] = tok2id[MASK]
    return X, y

encoder   = Encoder(V)
mlm_head  = MLMHead(V)
opt       = torch.optim.AdamW(list(encoder.parameters()) + list(mlm_head.parameters()), lr=3e-3)
loss_fn   = nn.CrossEntropyLoss(ignore_index=-100)

print("Pre-training encoder via MLM for 3000 steps...")
for step in range(3000):
    idx       = torch.randint(0, N_HOUSES, (128,))
    xb, yb    = mlm_mask(X[idx])
    h         = encoder(xb)
    logits    = mlm_head(h)
    loss      = loss_fn(logits.reshape(-1, V), yb.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()    # the 5-line loop from L1!
    if step % 500 == 0:
        print(f"  step {step:4d}   MLM loss {loss.item():.3f}")

pretrained_encoder = copy.deepcopy(encoder)
print("\nDone. Pre-trained encoder cached as `pretrained_encoder`.")

## 7 — The big experiment: PRE-TRAINING recipe vs BASELINE recipe

Let's be **very explicit** about what we're comparing.

```
   PRE-TRAINING RECIPE                       BASELINE RECIPE
   ─────────────────────                     ───────────────────
                                             (skip pre-training entirely)
   Step 1: Train encoder via MLM
           on all 8,000 houses
           (no price labels needed)
                                             Step 1: Random-init encoder
                                                     (knobs start as noise)

   Step 2: FREEZE encoder
           (don't nudge its knobs anymore)

   Step 3: Add price head                    Step 2: Add price head

   Step 4: Train price head ONLY             Step 3: Train EVERYTHING
           on N labelled houses                       (encoder + head together)
                                                      on N labelled houses
```

**Why compare these?** To answer: *does the pre-training step actually help?* If the baseline (skip pre-training, train end-to-end on labels) does just as well, then pre-training was wasted effort. If the pre-trained recipe wins — especially when labels are scarce — that's the whole foundation-model pitch validated.

Both recipes use the same data, same architecture, same downstream training loop. The only difference is whether we did the self-supervised pre-training step first.

In [ ]:
perm  = torch.randperm(N_HOUSES)
split = int(N_HOUSES * 0.8)
tr_idx, te_idx = perm[:split], perm[split:]
X_te, y_te     = X[te_idx], y[te_idx]

def freeze(mod):
    for p in mod.parameters(): p.requires_grad = False
    mod.eval()

def train_classifier(encoder, X_tr, y_tr, epochs=500, freeze_encoder=True, batch_size=128):
    head = PriceHead()
    if freeze_encoder:
        freeze(encoder)
        params = list(head.parameters())
    else:
        params = list(encoder.parameters()) + list(head.parameters())
    opt = torch.optim.AdamW(params, lr=3e-3)
    loss_fn = nn.CrossEntropyLoss()
    n = X_tr.size(0)
    for _ in range(epochs):
        if n > batch_size:
            idx = torch.randperm(n)[:batch_size]
            xb, yb = X_tr[idx], y_tr[idx]
        else:
            xb, yb = X_tr, y_tr
        h      = encoder(xb)
        logits = head(h)
        loss   = loss_fn(logits, yb)
        opt.zero_grad(); loss.backward(); opt.step()
    head.eval()
    with torch.no_grad():
        logits = head(encoder(X_te))
        pred = logits.argmax(-1)
        acc  = (pred == y_te).float().mean().item()
        per_class = []
        for c in range(5):
            mask = (y_te == c)
            if mask.sum() == 0:
                per_class.append(float("nan"))
            else:
                per_class.append((pred[mask] == c).float().mean().item())
        valid = [r for r in per_class if r == r]
        macro_recall = sum(valid) / len(valid)
    return acc, macro_recall, head

print(f"{'labels':>7} | {'pretrained acc':>14}  {'pretrained recall':>17} | "
      f"{'baseline acc':>12}  {'baseline recall':>15}")
print("-" * 84)
for n_labels in [50, 100, 500, 4000]:
    sub = tr_idx[:n_labels]
    X_tr, y_tr = X[sub], y[sub]
    enc_a = copy.deepcopy(pretrained_encoder)
    acc_a, rec_a, _ = train_classifier(enc_a, X_tr, y_tr, freeze_encoder=True)
    torch.manual_seed(n_labels)
    enc_b = Encoder(V)
    acc_b, rec_b, _ = train_classifier(enc_b, X_tr, y_tr, freeze_encoder=False)
    print(f"{n_labels:>7} | {acc_a:>14.3f}  {rec_a:>17.3f} | {acc_b:>12.3f}  {rec_b:>15.3f}")

## 8 — Interpreting the results

Look across the rows above:

- **50 labels**: PRE-TRAINING wins by ~7 accuracy points. With so few labels, training the encoder from scratch can't find the price-relevant patterns. The pre-trained encoder already knows the house structure — the price head just has to learn the small mapping from encoder output → price class.
- **100 labels**: PRE-TRAINING still ahead, smaller margin.
- **500 labels**: BASELINE overtakes. With more labels, end-to-end training can find encoder representations specifically optimised for price prediction.
- **4000 labels**: BASELINE wins clearly. End-to-end has enough signal to dominate.

### Why isn't pre-training a clear winner everywhere?

1. **Tabular data has limited contextual richness.** Only 10 attributes per house. Compare to L5b: 15 events × 4 attributes = 60 correlated pieces of info plus temporal ordering. More structure = more for pre-training to learn.
2. **A frozen encoder is a hard constraint.** It can't be tuned for the price task. Once we have enough labels, the baseline's flexibility wins.

**Real-world fix**: instead of a fully frozen probe, use **LoRA fine-tuning** (PRAGMA §3.1.2) — unfreeze ~2-4% of weights during downstream training. Pre-training's warm start AND tunability. Best of both worlds. We'll see this in the capstone.

## 9 — Sample predictions

Hand-craft a few houses and see what the model predicts.

In [ ]:
enc_final = copy.deepcopy(pretrained_encoder)
freeze(enc_final)
_, _, head_final = train_classifier(enc_final, X[tr_idx], y[tr_idx], epochs=1000, freeze_encoder=True)

demo_houses = [
    ("rural cottage", {
        "bedrooms":"2bed", "bathrooms":"1bath", "size":"small", "age":"vintage",
        "neighborhood":"rural", "garage":"1car", "pool":"nopool", "garden":"largegarden",
        "schools":"avgschool", "condition":"faircond"}),
    ("suburban family", {
        "bedrooms":"3bed", "bathrooms":"2bath", "size":"medium", "age":"modern",
        "neighborhood":"suburb", "garage":"2car", "pool":"nopool", "garden":"smallgarden",
        "schools":"goodschool", "condition":"goodcond"}),
    ("luxury beach", {
        "bedrooms":"5+bed", "bathrooms":"3+bath", "size":"huge", "age":"new",
        "neighborhood":"beach", "garage":"2car", "pool":"haspool", "garden":"largegarden",
        "schools":"excschool", "condition":"exccond"}),
]
for label, h in demo_houses:
    ids = torch.tensor([encode_house(h)])
    with torch.no_grad():
        logits = head_final(enc_final(ids))
        probs  = torch.softmax(logits, dim=-1)[0].tolist()
        pred   = logits.argmax(-1).item()
    bars = "  ".join(f"{PRICE_CLASSES[c]:>10s}={probs[c]*100:5.1f}%" for c in range(5))
    print(f"\n{label:18s}  prediction: {PRICE_CLASSES[pred]}")
    print(f"  {bars}")

## 10 — Things to try

1. **Drop the correlations.** Replace `random_house` with one that samples each attribute uniformly (ignoring archetypes). Re-run pre-training. Does pre-training still help at 50 labels? (Hint: probably not — there's nothing context-dependent for MLM to learn.)

2. **Inspect the pre-trained embeddings:**
```python
import torch.nn.functional as F
e_beach = pretrained_encoder.emb.weight[tok2id["beach"]]
e_pool  = pretrained_encoder.emb.weight[tok2id["haspool"]]
e_rural = pretrained_encoder.emb.weight[tok2id["rural"]]
print("beach · haspool:", F.cosine_similarity(e_beach.unsqueeze(0), e_pool.unsqueeze(0)).item())
print("rural · haspool:", F.cosine_similarity(e_rural.unsqueeze(0), e_pool.unsqueeze(0)).item())
```
After training, "beach" and "haspool" should be more similar than "rural" and "haspool" — because the model learned beach houses tend to have pools.

3. **More layers.** Bump `N_LAYERS` to 4 in the Encoder. Does pre-training help more?

## What's next

[Lesson 5b](../05b_walkthrough.md) — same recipe, applied to sequential event data, where pre-training wins much more decisively.
